In [7]:
!pip install google-api-python-client pandas matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from googleapiclient.discovery import build
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("=" * 55)
print(" SOCIAL MEDIA ENGAGEMENT DASHBOARD")
print(" YouTube Data API v3")
print("=" * 55)

API_KEY = input("Enter your youtube API key: ")

youtube = build('youtube', 'v3', developerKey=API_KEY)
CHANNELS = {

    "T-Series": "UCq-Fj5jknLsUf-MWSy4_brA"
}


def get_channel_stats(channel_id, channel_name):
    try:
        request = youtube.channels().list(
            part="snippet,statistics",
            id=channel_id
        )
        response = request.execute()
        if response['items']:
            stats = response['items'][0]['statistics']
            return {
                'channel_name': channel_name,
                'subscribers': int(stats.get('subscriberCount', 0)),
                'total_views': int(stats.get('viewCount', 0)),
                'total_videos': int(stats.get('videoCount', 0))
            }
    except Exception as e:
        print(f"[ERROR] {channel_name}: {e}")
    return None


def get_video_data(channel_id, channel_name, max_videos=15):
    videos = []
    try:
        search_request = youtube.search().list(
            part="id,snippet",
            channelId=channel_id,
            maxResults=max_videos,
            order="date",
            type="video"
        )
        search_response = search_request.execute()
        video_ids = [item['id']['videoId'] for item in search_response['items']]

        stats_request = youtube.videos().list(
            part="snippet,statistics",
            id=','.join(video_ids)
        )
        stats_response = stats_request.execute()

        for item in stats_response['items']:
            stats = item.get('statistics', {})
            snippet = item.get('snippet', {})

            pub_date = snippet.get('publishedAt', '')
            if pub_date:
                pub_date = datetime.strptime(pub_date, '%Y-%m-%dT%H:%M:%SZ')
                hour = pub_date.hour
                day = pub_date.strftime('%A')
                month = pub_date.strftime('%b %Y')
            else:
                hour, day, month = 0, 'Unknown', 'Unknown'

            likes = int(stats.get('likeCount', 0))
            comments = int(stats.get('commentCount', 0))
            views = int(stats.get('viewCount', 0))

            engagement_rate = ((likes + comments) / views * 100) if views > 0 else 0

            videos.append({
                'channel_name': channel_name,
                'video_title': snippet.get('title', '')[:50],
                'published_date': pub_date,
                'post_hour': hour,
                'day_of_week': day,
                'month': month,
                'views': views,
                'likes': likes,
                'comments': comments,
                'engagement_rate': round(engagement_rate, 2)
            })
    except Exception as e:
        print(f"[ERROR] Videos for {channel_name}: {e}")
    return videos


print("\n[INFO] Collecting channel statistics...")
channel_stats = []
for name, cid in CHANNELS.items():
    print(f"  Fetching {name}...")
    stats = get_channel_stats(cid, name)
    if stats:
        channel_stats.append(stats)

channel_df = pd.DataFrame(channel_stats)
print(f" Channel stats collected: {len(channel_df)} channels")

print("\n[INFO] Collecting video data...")
all_videos = []
for name, cid in CHANNELS.items():
    print(f"  Fetching videos from {name}...")
    videos = get_video_data(cid, name, max_videos=8)
    all_videos.extend(videos)

video_df = pd.DataFrame(all_videos)
print(f" Videos collected: {len(video_df)} videos")



if not video_df.empty:
    print("\n" + "=" * 55)
    print(" ANALYSIS RESULTS")
    print("=" * 55)

    eng_by_channel = video_df.groupby('channel_name')['engagement_rate'].mean().sort_values(ascending=False)
    print("\n Average Engagement Rate by Channel:")
    for channel, rate in eng_by_channel.items():
        print(f"   {channel}: {rate:.2f}%")

    best_hours = video_df.groupby('post_hour')['engagement_rate'].mean().nlargest(5)
    print("\n Top 5 Best Posting Hours:")
    for hour, rate in best_hours.items():
        ampm = "AM" if hour < 12 else "PM"
        hour_display = hour if hour <= 12 else hour - 12
        hour_display = 12 if hour_display == 0 else hour_display
        print(f"   {hour_display}:00 {ampm} → {rate:.2f}% engagement")

    best_day = video_df.groupby('day_of_week')['engagement_rate'].mean().sort_values(ascending=False)
    print("\n Best Days to Post:")
    for day, rate in best_day.head(3).items():
        print(f"   {day}: {rate:.2f}%")

    best_channel = eng_by_channel.idxmax()
    best_hour = best_hours.index[0]

    print("\n" + "=" * 55)
    print(" RECOMMENDATIONS")
    print("=" * 55)

    print(f"""
 BEST CHANNEL (highest engagement): {best_channel}
 BEST TIME TO POST: {best_hour}:00
 BEST DAY TO POST: {best_day.idxmax()}

 .CONTENT STRATEGY:
→ Post at {best_hour}:00 on {best_day.idxmax()}s
→ Study {best_channel}'s content style
→ Use engaging thumbnails and titles
→ Respond to comments within 1 hour

 .TARGETS (Next 30 Days):
→ Increase engagement rate by 25%
→ Post 3-4 times per week
→ Focus on trending topics
""")


    print("\n[EXPORT] Saving data for Power BI...")
    video_df.to_csv('youtube_video_data.csv', index=False, encoding='utf-8-sig')
    channel_df.to_csv('youtube_channel_stats.csv', index=False, encoding='utf-8-sig')
    print("[EXPORT] Saved: youtube_video_data.csv")
    print("[EXPORT] Saved: youtube_channel_stats.csv")


    from google.colab import files
    files.download('youtube_video_data.csv')
    files.download('youtube_channel_stats.csv')

    print("\n" + "=" * 55)
    print(" PROJECT COMPLETE!")
    print("  - youtube_video_data.csv ← Downloaded to your computer")
    print("  - youtube_channel_stats.csv ← Downloaded to your computer")
    print("=" * 55)
else:
    print("\n[WARNING] No video data was collected. Cannot perform analysis or recommendations.")
    print("Please ensure your API key is valid and has the necessary permissions.")

 SOCIAL MEDIA ENGAGEMENT DASHBOARD
 YouTube Data API v3

[INFO] Collecting channel statistics...
  Fetching T-Series...
 Channel stats collected: 1 channels

[INFO] Collecting video data...
  Fetching videos from T-Series...
 Videos collected: 8 videos

 ANALYSIS RESULTS

 Average Engagement Rate by Channel:
   T-Series: 2.64%

 Top 5 Best Posting Hours:
   11:00 AM → 3.10% engagement
   2:00 PM → 2.90% engagement
   7:00 AM → 2.80% engagement
   5:00 AM → 2.47% engagement
   1:00 PM → 1.15% engagement

 Best Days to Post:
   Sunday: 2.85%
   Saturday: 1.15%

 RECOMMENDATIONS

 BEST CHANNEL (highest engagement): T-Series
 BEST TIME TO POST: 11:00
 BEST DAY TO POST: Sunday

 .CONTENT STRATEGY:
→ Post at 11:00 on Sundays
→ Study T-Series's content style
→ Use engaging thumbnails and titles
→ Respond to comments within 1 hour

 .TARGETS (Next 30 Days):
→ Increase engagement rate by 25%
→ Post 3-4 times per week
→ Focus on trending topics


[EXPORT] Saving data for Power BI...
[EXPORT] Sav

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 PROJECT COMPLETE!
  - youtube_video_data.csv ← Downloaded to your computer
  - youtube_channel_stats.csv ← Downloaded to your computer
